In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install openpyxl pandas -q


Mounted at /content/drive


In [ ]:
from google.colab import files

print("Upload CP-AnemiC zip file:")
uploaded_cpanemic = files.upload()   # select your CP-AnemiC zip

print("\nUpload Eyes-Defy-Anemia zip file (or files, if split):")
uploaded_eda = files.upload()        # select your Eyes-Defy-Anemia zip

Upload CP-AnemiC zip file:


Saving CP-AnemiC (A Conjunctival Pallor) Dataset from Ghana.zip to CP-AnemiC (A Conjunctival Pallor) Dataset from Ghana.zip

Upload Eyes-Defy-Anemia zip file (or files, if split):


Saving archive.zip to archive.zip


In [ ]:
import zipfile, os

def unzip_all(zip_path, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    print(f"Extracting {zip_path} ...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_to)
    print(f"Done. Contents of {extract_to}:")
    for item in os.listdir(extract_to):
        print(" -", item)

unzip_all("/content/CP-AnemiC (A Conjunctival Pallor) Dataset from Ghana.zip", "/content/cpanemic_extracted")
unzip_all("/content/archive.zip", "/content/eda_extracted")

Extracting /content/CP-AnemiC (A Conjunctival Pallor) Dataset from Ghana.zip ...
Done. Contents of /content/cpanemic_extracted:
 - CP-AnemiC (A Conjunctival Pallor) Dataset from Ghana
Extracting /content/archive.zip ...
Done. Contents of /content/eda_extracted:
 - dataset anemia


In [ ]:
!apt-get install -y unrar -q

import glob
rar_files = glob.glob("/content/cpanemic_extracted/**/*.rar", recursive=True)
print("Found RAR files:", rar_files)

for rar_path in rar_files:
    rar_dir = os.path.dirname(rar_path)
    !unrar x -o+ "{rar_path}" "{rar_dir}/"

Reading package lists...
Building dependency tree...
Reading state information...
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Found RAR files: ['/content/cpanemic_extracted/CP-AnemiC (A Conjunctival Pallor) Dataset from Ghana/CP-AnemiC dataset.rar']

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/cpanemic_extracted/CP-AnemiC (A Conjunctival Pallor) Dataset from Ghana/CP-AnemiC dataset.rar

Creating    /content/cpanemic_extracted/CP-AnemiC (A Conjunctival Pallor) Dataset from Ghana/Anemic  OK
Extracting  /content/cpanemic_extracted/CP-AnemiC (A Conjunctival Pallor) Dataset from Ghana/Anemic/Image_001.png       0%  OK 
Extracting  /content/cpanemic_extracted/CP-AnemiC (A Conjunctival Pallor) Dataset from Ghana/Anemic/Image_002.png       0%  OK 
Extracting  /content/cpanemic_extracted/CP-AnemiC (A Conjunctival Pallor) Dataset from Ghana/Anemic

In [ ]:
def find_root(base_dir, marker_file_or_folder):
    for root, dirs, files_ in os.walk(base_dir):
        if marker_file_or_folder in dirs or marker_file_or_folder in files_:
            return root
    return None

CPANEMIC_ROOT = find_root("/content/cpanemic_extracted", "Anemia_Data_Collection_Sheet.xlsx")
print("CP-AnemiC root:", CPANEMIC_ROOT)

EDA_ROOT = find_root("/content/eda_extracted", "Italy")
print("Eyes-Defy-Anemia root:", EDA_ROOT)

CP-AnemiC root: /content/cpanemic_extracted/CP-AnemiC (A Conjunctival Pallor) Dataset from Ghana
Eyes-Defy-Anemia root: /content/eda_extracted/dataset anemia


In [ ]:
EDA_ITALY_DIR = os.path.join(EDA_ROOT, "Italy")
EDA_INDIA_DIR = os.path.join(EDA_ROOT, "India")

print("Italy dir exists:", os.path.exists(EDA_ITALY_DIR))
print("India dir exists:", os.path.exists(EDA_INDIA_DIR))
print("Italy.xlsx exists:", os.path.exists(os.path.join(EDA_ITALY_DIR, "Italy.xlsx")))
print("India.xlsx exists:", os.path.exists(os.path.join(EDA_INDIA_DIR, "India.xlsx")))

Italy dir exists: True
India dir exists: True
Italy.xlsx exists: True
India.xlsx exists: True


In [ ]:
def severity_child_6_59mo(hb):
    """WHO cutoffs for children 6-59 months."""
    if hb >= 11.0:
        return "Non-Anemic"
    elif hb >= 10.0:
        return "Mild"
    elif hb >= 7.0:
        return "Moderate"
    else:
        return "Severe"


def severity_adult(hb, sex):
    """WHO cutoffs for non-pregnant adults (>=15y), sex-specific."""
    sex = (sex or "").strip().upper()
    if sex == "F":
        if hb >= 12.0:
            return "Non-Anemic"
        elif hb >= 11.0:
            return "Mild"
        elif hb >= 8.0:
            return "Moderate"
        else:
            return "Severe"
    elif sex == "M":
        if hb >= 13.0:
            return "Non-Anemic"
        elif hb >= 11.0:
            return "Mild"
        elif hb >= 8.0:
            return "Moderate"
        else:
            return "Severe"
    else:
        return "Unknown"


SEVERITY_ORDER = {"Non-Anemic": 0, "Mild": 1, "Moderate": 2, "Severe": 3}

In [ ]:
!pip install openpyxl pandas -q

In [ ]:
import openpyxl
import pandas as pd

def load_cpanemic(root_dir):
    xlsx_path = os.path.join(root_dir, "Anemia_Data_Collection_Sheet.xlsx")
    wb = openpyxl.load_workbook(xlsx_path, data_only=True)
    ws = wb[wb.sheetnames[0]]

    rows = []
    for r in ws.iter_rows(min_row=2, values_only=True):
        if r[0] is None:
            continue
        image_id, hb, severity, age_months, gender = r[0], r[1], r[2], r[3], r[4]
        remark = r[5] if len(r) > 5 else None

        subfolder = "Anemic" if str(remark).strip().lower() == "anemic" else "Non-anemic"
        image_path = os.path.join(root_dir, subfolder, f"{image_id}.png")

        rows.append({
            "image_id": image_id,
            "source_dataset": "CP-AnemiC",
            "subject_id": f"CPA_{image_id}",
            "hb_value": hb,
            "severity_class": severity,
            "severity_rank": SEVERITY_ORDER.get(severity),
            "age_years": round(age_months / 12, 2) if isinstance(age_months, (int, float)) else None,
            "age_group": "child_6_59mo",
            "sex": gender,
            "population": "Ghana",
            "has_mask": False,
            "image_path": image_path,
            "mask_path": None,
        })

    df = pd.DataFrame(rows)
    missing = df[~df["image_path"].apply(os.path.exists)]
    print(f"[CP-AnemiC] loaded {len(df)} rows, {len(missing)} missing image files")
    return df

cpanemic_df = load_cpanemic(CPANEMIC_ROOT)
cpanemic_df.head()

[CP-AnemiC] loaded 710 rows, 0 missing image files


,image_id,source_dataset,subject_id,hb_value,severity_class,severity_rank,age_years,age_group,sex,population,has_mask,image_path,mask_path
0,Image_001,CP-AnemiC,CPA_Image_001,9.8,Moderate,2,0.5,child_6_59mo,Female,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None
1,Image_002,CP-AnemiC,CPA_Image_002,9.9,Moderate,2,2.0,child_6_59mo,Male,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None
2,Image_003,CP-AnemiC,CPA_Image_003,11.1,Non-Anemic,0,2.0,child_6_59mo,Female,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None
3,Image_004,CP-AnemiC,CPA_Image_004,12.5,Non-Anemic,0,1.0,child_6_59mo,Male,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None
4,Image_005,CP-AnemiC,CPA_Image_005,9.9,Moderate,2,2.0,child_6_59mo,Male,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None


In [ ]:
def load_eyes_defy_subset(xlsx_path, image_root, population_label):
    wb = openpyxl.load_workbook(xlsx_path, data_only=True)
    ws = wb[wb.sheetnames[0]]

    rows = []
    n_missing_hb = 0
    for r in ws.iter_rows(min_row=2, values_only=True):
        if r[0] is None:
            continue
        number, hb, sex, age, note = r[0], r[1], r[2], r[3], r[4]

        if not isinstance(hb, (int, float)):
            n_missing_hb += 1
            continue

        number = int(number)
        folder = os.path.join(image_root, str(number))
        palpebral_path = os.path.join(folder, "palpebral.png")

        # original whole-eye jpg, kept for reference only (not used for training)
        original_jpg = None
        if os.path.isdir(folder):
            for fname in os.listdir(folder):
                if fname.lower().endswith(".jpg"):
                    original_jpg = os.path.join(folder, fname)
                    break

        severity = severity_adult(hb, sex)

        rows.append({
            "image_id": f"{population_label}_{number}",
            "source_dataset": "Eyes-Defy-Anemia",
            "subject_id": f"EDA_{population_label}_{number}",
            "hb_value": hb,
            "severity_class": severity,
            "severity_rank": SEVERITY_ORDER.get(severity),
            "age_years": age,
            "age_group": "adult",
            "sex": sex,
            "population": population_label,
            "has_mask": os.path.exists(palpebral_path),
            "image_path": palpebral_path if os.path.exists(palpebral_path) else None,  # <-- USE palpebral crop as the training image
            "original_whole_eye_path": original_jpg,  # kept for reference, not used for training
            "note": note,
        })

    df = pd.DataFrame(rows)
    print(f"[Eyes-Defy-Anemia / {population_label}] {len(df)} usable rows, {n_missing_hb} excluded (missing Hb)")
    return df

In [ ]:
unified = pd.concat([cpanemic_df, eda_df], ignore_index=True, sort=False)

print("Total rows:", len(unified))
print("\nBy source dataset:")
print(unified["source_dataset"].value_counts())
print("\nBy severity class:")
print(unified["severity_class"].value_counts())
print("\nCross-tab (source x severity):")
print(pd.crosstab(unified["source_dataset"], unified["severity_class"]))

unified.head()

Total rows: 909

By source dataset:
source_dataset
CP-AnemiC           710
Eyes-Defy-Anemia    199
Name: count, dtype: int64

By severity class:
severity_class
Non-Anemic    395
Moderate      283
Mild          180
Severe         51
Name: count, dtype: int64

Cross-tab (source x severity):
severity_class    Mild  Moderate  Non-Anemic  Severe
source_dataset                                      
CP-AnemiC          144       232         286      48
Eyes-Defy-Anemia    36        51         109       3


,image_id,source_dataset,subject_id,hb_value,severity_class,severity_rank,age_years,age_group,sex,population,has_mask,image_path,mask_path,has_forniceal_mask,note
0,Image_001,CP-AnemiC,CPA_Image_001,9.8,Moderate,2,0.5,child_6_59mo,Female,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None,NaN,NaN
1,Image_002,CP-AnemiC,CPA_Image_002,9.9,Moderate,2,2.0,child_6_59mo,Male,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None,NaN,NaN
2,Image_003,CP-AnemiC,CPA_Image_003,11.1,Non-Anemic,0,2.0,child_6_59mo,Female,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None,NaN,NaN
3,Image_004,CP-AnemiC,CPA_Image_004,12.5,Non-Anemic,0,1.0,child_6_59mo,Male,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None,NaN,NaN
4,Image_005,CP-AnemiC,CPA_Image_005,9.9,Moderate,2,2.0,child_6_59mo,Male,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None,NaN,NaN


In [ ]:
ct = pd.crosstab(unified["source_dataset"], unified["severity_class"], normalize="columns")
for col in ct.columns:
    max_share = ct[col].max()
    if max_share > 0.9:
        dominant_source = ct[col].idxmax()
        print(f"⚠ WARNING: class '{col}' is {max_share:.0%} from source '{dominant_source}' "
              f"— report this class's metrics separately by source.")

⚠ WARNING: class 'Severe' is 94% from source 'CP-AnemiC' — report this class's metrics separately by source.


In [ ]:
missing_images = unified[unified["image_path"].isna() | ~unified["image_path"].apply(lambda p: os.path.exists(p) if p else False)]
print(f"{len(missing_images)} rows have no resolvable image file")
missing_images[["image_id", "source_dataset", "image_path"]].head(20)

0 rows have no resolvable image file


,image_id,source_dataset,image_path


In [ ]:
import shutil

OUTPUT_ROOT = "/content/unified_dataset"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

for cls in unified["severity_class"].unique():
    os.makedirs(os.path.join(OUTPUT_ROOT, cls), exist_ok=True)

copied = 0
skipped = 0
new_paths = []

for _, row in unified.iterrows():
    src = row["image_path"]
    cls = row["severity_class"]

    if not src or not os.path.exists(src):
        skipped += 1
        new_paths.append(None)
        continue

    ext = os.path.splitext(src)[1]
    dest_filename = f"{row['source_dataset']}_{row['image_id']}{ext}"
    dest_path = os.path.join(OUTPUT_ROOT, cls, dest_filename)

    shutil.copy2(src, dest_path)
    new_paths.append(dest_path)
    copied += 1

unified["final_image_path"] = new_paths

print(f"Copied {copied} images, skipped {skipped} (missing files)")

Copied 909 images, skipped 0 (missing files)


In [ ]:
for cls in unified["severity_class"].unique():
    folder = os.path.join(OUTPUT_ROOT, cls)
    n_files = len(os.listdir(folder))
    print(f"{cls}: {n_files} images")

Moderate: 283 images
Non-Anemic: 395 images
Mild: 180 images
Severe: 51 images


In [ ]:
CSV_PATH = os.path.join(OUTPUT_ROOT, "unified_dataset.csv")
unified.to_csv(CSV_PATH, index=False)
print(f"Saved metadata CSV to {CSV_PATH}")
unified.head()

Saved metadata CSV to /content/unified_dataset/unified_dataset.csv


,image_id,source_dataset,subject_id,hb_value,severity_class,severity_rank,age_years,age_group,sex,population,has_mask,image_path,mask_path,has_forniceal_mask,note,final_image_path
0,Image_001,CP-AnemiC,CPA_Image_001,9.8,Moderate,2,0.5,child_6_59mo,Female,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None,NaN,NaN,/content/unified_dataset/Moderate/CP-AnemiC_Im...
1,Image_002,CP-AnemiC,CPA_Image_002,9.9,Moderate,2,2.0,child_6_59mo,Male,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None,NaN,NaN,/content/unified_dataset/Moderate/CP-AnemiC_Im...
2,Image_003,CP-AnemiC,CPA_Image_003,11.1,Non-Anemic,0,2.0,child_6_59mo,Female,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None,NaN,NaN,/content/unified_dataset/Non-Anemic/CP-AnemiC_...
3,Image_004,CP-AnemiC,CPA_Image_004,12.5,Non-Anemic,0,1.0,child_6_59mo,Male,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None,NaN,NaN,/content/unified_dataset/Non-Anemic/CP-AnemiC_...
4,Image_005,CP-AnemiC,CPA_Image_005,9.9,Moderate,2,2.0,child_6_59mo,Male,Ghana,False,/content/cpanemic_extracted/CP-AnemiC (A Conju...,None,NaN,NaN,/content/unified_dataset/Moderate/CP-AnemiC_Im...


In [ ]:
shutil.make_archive("/content/unified_dataset", 'zip', OUTPUT_ROOT)

from google.colab import files
files.download("/content/unified_dataset.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>